# 03 — Missing Values and Outliers

**Package:** `fraud_nb01-07_v2` (flat S3 layout, shared verified helpers)

**Phase 1, notebook 3 of 7.** The first notebook that **fits** anything.

Every statistic computed here — each median, each winsorising cap — is fitted on
`__split == 'train'` rows only and then applied to the whole frame. That boundary is enforced by
a leakage regression test in section 5 that fails the notebook if any stored statistic matches
the train+test value instead of the train-only value.

**Two decisions that shape everything downstream**

1. **Structural absence is not missingness.** 62.5% of payments have no `card_token` because a
   UPI payment has no card. Imputing a median BIN across 313,000 of them would invent data.
   These columns keep their nulls and gain a `has_card` indicator; tree models read a null as a
   missing-branch signal, which is exactly right here.
2. **Nothing is capped in place.** Fraud concentrates in the upper tail — capping
   `payment_amount` deletes the Stolen Card signature outright. Winsorised and log variants are
   created *alongside* the originals for the linear model, with caps fitted on train.

## 0. Colab bootstrap

Keys come from **Colab Secrets** (key icon, left sidebar) — `AWS_ACCESS_KEY_ID` and
`AWS_SECRET_ACCESS_KEY`, both with notebook access enabled. They are loaded into environment
variables so boto3 still resolves through the default credential chain, keeping the client
construction identical to what runs under IRSA in production. Nothing below prints a key.

In [ ]:
%pip install -q boto3==1.43.95

## 1. Configuration

In [ ]:
# MARKER: fraud_nb01-07_v2 :: 03_Missing_Values_and_Outliers
import io, os, json, time, hashlib, platform, importlib
from datetime import datetime, timezone
import boto3
from botocore.exceptions import ClientError
import joblib
import numpy as np
import pandas as pd
from google.colab import userdata

BUCKET, REGION = "fraud-ecommerce", "ap-south-2"
SPLIT_DATE = pd.Timestamp("2025-07-01")      # stamped in notebook 01 as __split; verified here, never recomputed
CONTRACT_VERSION = "v1"
SEED = 42
MARKER = "fraud_nb01-07_v2"
for _k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"):
    if not os.environ.get(_k):
        os.environ[_k] = userdata.get(_k)     # Colab Secrets -> process env only; never printed or saved
s3 = boto3.client("s3", region_name=REGION)
RAW, LABELS, CONTRACTS, DATA, REPORTS = "raw/", "raw/label_sources/", "contracts/", "data/", "reports/"  # flat layout
ident = boto3.client("sts", region_name=REGION).get_caller_identity()
print("account:", ident["Account"], "| arn:", ident["Arn"])
if ident["Arn"].endswith(":root"):
    print("NOTE: running as root — accepted for Phase 1; move to an IAM principal before Phase 2")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)


def _jsonable(o):
    if isinstance(o, (np.integer, np.floating, np.bool_)):
        return o.item()
    if isinstance(o, (pd.Timestamp, datetime)):
        return o.isoformat()
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(f"not JSON-serialisable: {type(o).__name__}")


def read_bytes_s3(key):
    return s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()


def put_bytes_s3(body, key):
    s3.put_object(Bucket=BUCKET, Key=key, Body=body)
    print(f"saved s3://{BUCKET}/{key}  ({len(body):,} bytes)")


def key_exists(key):
    try:
        s3.head_object(Bucket=BUCKET, Key=key)
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] in ("404", "NoSuchKey", "NotFound"):
            return False
        raise


def read_s3(key):
    return pd.read_parquet(io.BytesIO(read_bytes_s3(key)))


def save_s3(df, key):
    """Parquet only; the bytes are verified to round-trip columns, dtypes and categories before upload."""
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    body = buf.getvalue()
    back = pd.read_parquet(io.BytesIO(body))
    assert list(back.columns) == list(df.columns) and len(back) == len(df), f"parquet round-trip changed shape: {key}"
    bad = [c for c in df.columns if str(back[c].dtype) != str(df[c].dtype)]
    assert not bad, f"parquet round-trip changed dtypes in {key}: {bad}"
    badcat = [c for c in df.columns if str(df[c].dtype) == "category"
              and list(back[c].cat.categories) != list(df[c].cat.categories)]
    assert not badcat, f"parquet round-trip changed categories in {key}: {badcat}"
    put_bytes_s3(body, key)


def read_json_s3(key):
    return json.loads(read_bytes_s3(key))


def save_json_s3(obj, key):
    put_bytes_s3(json.dumps(obj, indent=1, default=_jsonable).encode(), key)


def save_model_s3(obj, key):
    buf = io.BytesIO()
    joblib.dump(obj, buf)
    put_bytes_s3(buf.getvalue(), key)


def load_model_s3(key):
    return joblib.load(io.BytesIO(read_bytes_s3(key)))


# names used by notebooks 01-04 (same verified implementations underneath)
def s3_read_csv(key, **kw):
    return pd.read_csv(io.BytesIO(read_bytes_s3(key)), **kw)


s3_read_parquet, s3_read_json = read_s3, read_json_s3


def s3_write_parquet(df, key):
    save_s3(df, key)
    return f"s3://{BUCKET}/{key}"


def s3_write_json(obj, key):
    save_json_s3(obj, key)
    return f"s3://{BUCKET}/{key}"


RAW_KEYS = ([f"{RAW}{t}.csv" for t in ["payments", "orders", "order_items", "account_logins", "customers",
                                         "merchants", "cards", "devices", "ip_reputation"]]
            + [f"{LABELS}{t}.csv" for t in ["fraud_events", "audit_sample", "chargebacks"]]
            + [f"{CONTRACTS}schema_v1.json"])
_absent = [k for k in RAW_KEYS if not key_exists(k)]
assert not _absent, f"missing landing objects in s3://{BUCKET}/: {_absent}"
print(f"landing objects present: {len(RAW_KEYS)} (flat layout, bucket root)")


def run_meta(notebook):
    return {"notebook": notebook, "marker": MARKER,
            "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "library_versions": LIB_VERSIONS, "version_drift": VERSION_DRIFT}


LIBS = ["pandas", "numpy", "pyarrow", "scipy", "statsmodels", "sklearn", "lightgbm", "xgboost", "joblib", "boto3"]
LIB_VERSIONS = {"python": platform.python_version(),
                **{m: importlib.import_module(m).__version__ for m in LIBS}}
EXPECTED = {"pandas": "2.2.3", "numpy": "2.1.3", "pyarrow": "23.0.1", "scipy": "1.16.3", "statsmodels": "0.15.0",
            "sklearn": "1.6.1", "lightgbm": "4.6.0", "xgboost": "3.4.1", "boto3": "1.43.95"}
VERSION_DRIFT = {m: {"verified": v, "found": LIB_VERSIONS[m]} for m, v in EXPECTED.items() if LIB_VERSIONS[m] != v}
if not LIB_VERSIONS["python"].startswith("3.13."):
    VERSION_DRIFT["python"] = {"verified": "3.13.x", "found": LIB_VERSIONS["python"]}
print(LIB_VERSIONS)
print("VERSION DRIFT vs the runtime verified on 2026-09-16:", VERSION_DRIFT or "none")

## 2. Load stage 02 and establish the fit boundary

In [ ]:
df = s3_read_parquet(f'{DATA}02_cleaned.parquet')
TRAIN = df.__split == 'train'

print(f'loaded {df.shape[0]:,} x {df.shape[1]}')
print(f'train {int(TRAIN.sum()):,} | test {int((~TRAIN).sum()):,}')
print('\nEVERY .fit() / .median() / .quantile() below is keyed off TRAIN.')

nulls = df.isna().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)
print(f'\n{len(nulls)} columns carry nulls:')
print(pd.DataFrame({'nulls': nulls,
                    'pct': (100 * nulls / len(df)).round(2)}).head(25).to_string())

## 3. Classify the nulls

Four groups, four different answers. Lumping them into one imputer is how a pipeline ends up
asserting that the median customer holds a card.

| Group | Cause | Treatment |
|---|---|---|
| Card columns | non-card payment method — **structural** | keep null, add `has_card` |
| Device columns | fingerprint service outage window | keep null, add `has_device_profile` |
| Basket columns | 380 orders with no line items | keep null, add `has_basket` — **not zero** |
| Login columns | no login precedes this payment | keep null, add `has_prior_login` |

A zero would be a lie in every one of these cases: a basket of zero value is not the same thing
as a basket we have no rows for.

In [ ]:
CARD_COLS   = ['bin', 'issuer', 'network', 'product_type', 'issuing_country', 'is_prepaid',
               'token_first_seen_timestamp', 'card_token_age_h', 'issuer_foreign']
DEVICE_COLS = ['device_type', 'os_family', 'browser_family', 'is_emulator',
               'first_seen_timestamp', 'device_age_h', 'device_n_customers']
BASKET_COLS = ['n_lines', 'n_categories', 'max_unit_price', 'total_qty', 'basket_value']
LOGIN_COLS  = ['last_login_new_device', 'last_login_unusual_loc',
               'last_login_failed_attempts', 'last_login_risk', 'hours_since_last_login']

df['has_card']           = df.card_token.notna().astype(int)
df['has_device_profile'] = df.device_type.notna().astype(int)
df['has_basket']         = df.n_lines.notna().astype(int)
df['has_prior_login']    = df.hours_since_last_login.notna().astype(int)

for name, cols, ind in [('card', CARD_COLS, 'has_card'),
                        ('device', DEVICE_COLS, 'has_device_profile'),
                        ('basket', BASKET_COLS, 'has_basket'),
                        ('login', LOGIN_COLS, 'has_prior_login')]:
    print(f'{name:8s} present on {100*df[ind].mean():6.2f}% of rows '
          f'({int(df[ind].sum()):,}) — {len(cols)} columns kept null on the rest')

## 4. Imputation, fitted on train only

Group-wise where a group genuinely explains the value (shipping-address age varies by shipping
speed; account age and return behaviour vary by city tier), global median otherwise, explicit
`Unknown` level for categoricals.

`Unknown` is a level, never a null. A missing category that stays null silently disappears from
a one-hot matrix and reappears as an all-zero row, which is not the same thing as a category the
model can learn about.

In [ ]:
IMPUTE_NUM_BY_GROUP = {'shipping_addr_age_hours': 'shipping_speed',
                       'account_age_days':        'city_tier',
                       'prior_return_rate':       'city_tier',
                       'prior_orders_12m':        'city_tier'}
# Region codes are identifiers, not quantities: the median of {11, 56, 80} is not a
# plausible region, it is a region that does not describe the row. Filled with -1 as an
# explicit "unknown region" level instead.
REGION_CODES = ['home_region_code', 'ip_region_code']
IMPUTE_NUM_GLOBAL = ['kyc_level', 'city_tier',
                     'attempt_seq_in_session', 'avg_ticket_size',
                     'trailing_chargeback_rate_bps', 'reputation_score', 'is_hosting',
                     'address_match_flag', 'bill_ship_mismatch', 'order_value',
                     'discount_amount', 'shipping_charge', 'item_count',
                     'ip_country_mismatch', 'ip_region_mismatch']
IMPUTE_CAT = ['payment_gateway', 'shipping_speed', 'delivery_type', 'city',
              'email_domain_class', 'acquisition_channel', 'merchant_category',
              'asn_type', 'ip_country', 'shipping_pincode', 'home_pincode']

fitted = {'group_medians': {}, 'global_medians': {}, 'sentinel_fill': {}, 'cat_fill': {}}

# Group KEYS must be clean before they are grouped on. city_tier and shipping_speed are
# both keys below, so they are imputed first — otherwise every null-keyed row silently
# falls through to the global fallback instead of its group's median.
for key in ['city_tier', 'shipping_speed']:
    if df[key].isna().any():
        if df[key].dtype.kind in 'fiu':
            v = float(df.loc[TRAIN, key].median())
            fitted['global_medians'][key] = v
            df[key] = df[key].fillna(v)
        else:
            fitted['cat_fill'][key] = 'Unknown'
            df[key] = df[key].astype(object).where(df[key].notna(), 'Unknown')
        print(f'group key {key} imputed first')

for col, by in IMPUTE_NUM_BY_GROUP.items():
    med = df.loc[TRAIN].groupby(by, dropna=False)[col].median()      # <-- TRAIN ONLY
    fitted['group_medians'][col] = {
        'by': by,
        'map': {str(k): (None if pd.isna(v) else float(v)) for k, v in med.items()},
        'fallback': float(df.loc[TRAIN, col].median())}
    df[col] = (df[col].fillna(df[by].map(med).astype('float64'))
                      .fillna(fitted['group_medians'][col]['fallback']))

for col in IMPUTE_NUM_GLOBAL:
    v = df.loc[TRAIN, col].median()                                   # <-- TRAIN ONLY
    if pd.isna(v):
        continue
    fitted['global_medians'][col] = float(v)
    df[col] = df[col].fillna(v)

for col in REGION_CODES:
    fitted['sentinel_fill'][col] = -1.0           # explicit 'unknown region' level, NOT a median
    df[col] = df[col].fillna(-1.0)

for col in IMPUTE_CAT:
    fitted['cat_fill'][col] = 'Unknown'
    df[col] = df[col].astype(object).where(df[col].notna(), 'Unknown')

print(f"{len(fitted['group_medians'])} group medians, "
      f"{len(fitted['global_medians'])} global medians, "
      f"{len(fitted['cat_fill'])} Unknown levels — all fitted on train rows only")

## 5. Leakage regression test

A `fit`/`transform` signature proves nothing. This test proves the **call site** used train rows
only: it recomputes each statistic over train+test and asserts the stored value matches the
train-only figure instead.

Watch `account_age_days` — the two differ by roughly three months. If imputation had been fitted
on the full frame, every training row would carry a value computed partly from the future, and
nothing downstream would have flagged it.

In [ ]:
src = s3_read_parquet(f'{DATA}02_cleaned.parquet')

print('statistic            train-only        train+test    stored == train-only')
print('-' * 72)
for col in ['account_age_days', 'order_value', 'prior_return_rate', 'avg_ticket_size']:
    tr_only  = src.loc[src.__split == 'train', col].median()
    all_rows = src[col].median()
    stored = (fitted['group_medians'].get(col, {}).get('fallback')
              if col in fitted['group_medians'] else fitted['global_medians'].get(col))
    match = stored is not None and abs(stored - tr_only) < 1e-9
    print(f'{col:20s} {tr_only:12.4f} {all_rows:15.4f}    {match}')
    assert match, f'{col} was fitted on more than the training split'

del src
print('\nPASS — every stored statistic equals its train-only value')

## 6. Outliers

The IQR rule flags 9–14% of rows on the money columns. That is not an outlier rate, it is a
lognormal distribution meeting a rule designed for symmetric ones.

**Nothing is capped in place.** Fraudulent payments have a median of about twice the legitimate
median and a 99th percentile roughly three times higher — the signal *is* the tail. Winsorised
(`_w995`) and log variants are created alongside the originals so notebook 06 can give the
linear model bounded inputs while the trees keep the raw values.

The ~2,000 accounts with 300+ prior orders are wholesale customers. They are a real segment, not
data errors, and are left untouched.

In [ ]:
SKEWED = ['payment_amount', 'basket_value', 'max_unit_price', 'amount_vs_merchant_ticket',
          'device_n_customers', 'ip_n_customers', 'prior_orders_12m', 'processing_fee']

rep = []
for col in SKEWED:
    s = df.loc[TRAIN, col].dropna()                                   # <-- TRAIN ONLY
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    rep.append({'column': col, 'p50': round(s.median(), 2), 'p99': round(s.quantile(.99), 2),
                'max': round(s.max(), 2), 'iqr_upper': round(q3 + 1.5 * iqr, 2),
                'pct_flagged': round(100 * (s > q3 + 1.5 * iqr).mean(), 2),
                'skew': round(s.skew(), 2)})
print(pd.DataFrame(rep).to_string(index=False))

WINSOR = ['payment_amount', 'basket_value', 'max_unit_price', 'amount_vs_merchant_ticket']
fitted['winsor_p995'] = {}
for col in WINSOR:
    cap = float(df.loc[TRAIN, col].quantile(.995))                    # <-- TRAIN ONLY
    fitted['winsor_p995'][col] = cap
    df[col + '_w995'] = df[col].clip(upper=cap)
    df['log_' + col]  = np.log1p(df[col].clip(lower=0))

print(f'\ncaps fitted on train: {({k: round(v) for k, v in fitted["winsor_p995"].items()})}')
print('originals retained UNCAPPED — fraud concentrates in the upper tail')
print(f'wholesale accounts (>300 prior orders) retained: '
      f'{int((df.prior_orders_12m > 300).sum()):,}')

## 7. NaN policy

Two classes, asserted separately.

- **Non-structural numerics must be null-free.** If any survive, imputation missed a column and
  the assertion fails the notebook rather than letting a silent `NaN` reach `StandardScaler`,
  which computes statistics ignoring nulls and then propagates them.
- **Structural nulls are retained deliberately**, each with an indicator column beside it.

`account_age_days_at_txn` is dropped: it is superseded by the imputed `account_age_days`, and
keeping both would carry 4,964 sentinel nulls forward into the model.

In [ ]:
STRUCTURAL_NULL = set(CARD_COLS + DEVICE_COLS + BASKET_COLS + LOGIN_COLS)
STRUCTURAL_NULL |= {f'{c}_w995' for c in STRUCTURAL_NULL} | {f'log_{c}' for c in STRUCTURAL_NULL}
STRUCTURAL_NULL |= {'account_age_days_at_txn'}

must_be_clean = [c for c in df.columns
                 if c not in STRUCTURAL_NULL and df[c].dtype.kind in 'fiub'
                 and not c.startswith(('token_', 'first_seen', 'signup'))]
offenders = {c: int(df[c].isna().sum()) for c in must_be_clean if df[c].isna().any()}
print('non-structural numeric columns:', 'CLEAN' if not offenders else offenders)
assert not offenders, f'unexpected nulls survived imputation: {offenders}'

df = df.drop(columns=['account_age_days_at_txn'])

remaining = df[sorted(STRUCTURAL_NULL & set(df.columns))].isna().sum()
print('\nstructural nulls retained (indicator present, trees branch on them):')
print(remaining[remaining > 0].to_string())

## 8. Save stage 03

The fitted parameters are persisted alongside the data. Notebook 07 loads them into the preprocessor artifact, and Phase 2 serving applies exactly these values — not recomputed ones.

In [ ]:
assert df.payment_id.is_unique
assert df.__split.isin(['train', 'test']).all()

print(f'03_imputed: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(s3_write_parquet(df, f'{DATA}03_imputed.parquet'))
print(s3_write_json(fitted, f'{DATA}03_fitted_params.json'))

record = {**run_meta('03_Missing_Values_and_Outliers'),
          'rows': int(len(df)), 'cols': int(df.shape[1]),
          'group_medians': list(fitted['group_medians']),
          'global_medians': list(fitted['global_medians']),
          'sentinel_fill': fitted['sentinel_fill'],
          'unknown_levels': list(fitted['cat_fill']),
          'winsor_caps': fitted['winsor_p995'],
          'structural_null_columns': sorted(STRUCTURAL_NULL & set(df.columns)),
          'capping_policy': 'no in-place capping; fraud concentrates in the upper tail',
          'fit_boundary': "__split == 'train'"}
print(s3_write_json(record, f'{REPORTS}03_run_record.json'))